# GSM8K Reasoning Lab

Compare a base language model with a LoRA adapter on math problems. This clean walkthrough runs the project CLI in separate processes. Install the package and ML extra using the README first. No saved benchmark results are included.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from the cloned project")

def lab(*args):
    subprocess.run([sys.executable, "-m", "gsm8k_lab.cli", *args], cwd=ROOT, check=True)

## 1. Try the scoring demo

This uses synthetic answers, requires no model download, and is not a benchmark.

In [ ]:
lab("demo")

## 2. Configure a run

The remaining cells download Qwen2.5-1.5B-Instruct and GSM8K, then perform actual inference and training. Use suitable hardware. A 4 GB GPU is not a validated full-model training configuration. Keep RUN unique; existing outputs cannot be overwritten. Start with development data, and save the test set for your final configuration.

In [ ]:
RUN = "notebook-01"
TRAIN_ROWS = "1000"
EVAL_ROWS = "100"
MODEL_REVISION = "main"  # Replace with a saved commit hash to reproduce a run.
DATASET_REVISION = "main"

def folder(name):
    return f"runs/{RUN}/{name}"

def experiment(command, *args):
    lab(command, "--revision", MODEL_REVISION, "--dataset-revision", DATASET_REVISION, *args)

## 3. Evaluate the base model

In [ ]:
experiment("evaluate", "--split", "dev", "--limit", EVAL_ROWS, "--output", folder("base-dev"))

# Freeze the exact upstream versions for the rest of this notebook.
import json
baseline = json.loads((ROOT / folder("base-dev") / "summary.json").read_text(encoding="utf-8"))
MODEL_REVISION = baseline["model_revision"]
DATASET_REVISION = baseline["data"]["revision"]

## 4. Train and evaluate a LoRA adapter

The adapter changes a small subset of model parameters. Training and development rows do not overlap. Long examples are skipped and counted.

In [ ]:
experiment("train", "--limit", TRAIN_ROWS, "--output", folder("lora"))
experiment("evaluate", "--adapter", folder("lora/adapter"), "--split", "dev", "--limit", EVAL_ROWS, "--output", folder("lora-dev"))
lab("compare", folder("base-dev/summary.json"), folder("lora-dev/summary.json"))

## 5. Compare prompting

Three examples come from a dedicated demonstration pool. Change only one experimental setting at a time.

In [ ]:
experiment("evaluate", "--adapter", folder("lora/adapter"), "--shots", "3", "--limit", EVAL_ROWS, "--output", folder("lora-3shot-dev"))
lab("compare", folder("lora-dev/summary.json"), folder("lora-3shot-dev/summary.json"))

## 6. Final test evaluation

Run these cells after choosing your settings on development data. The example compares base and adapter with the standard zero-shot prompt.

In [ ]:
experiment("evaluate", "--split", "test", "--limit", EVAL_ROWS, "--output", folder("base-test"))
experiment("evaluate", "--adapter", folder("lora/adapter"), "--split", "test", "--limit", EVAL_ROWS, "--output", folder("lora-test"))
lab("compare", folder("base-test/summary.json"), folder("lora-test/summary.json"))

## Before sharing

Review local summaries and predictions, clear all notebook outputs, and remove private metadata. Generated artifacts are ignored by Git. Consult `docs/protocol.md` and `docs/sharing.md` before reporting or publishing results.